In [1]:
%load_ext autoreload
%autoreload 2  

In [2]:
import numpy as np
from src.game.connect_four import BoardState, ConnectFourGame
from src.mcts.tree import Node
from src.mcts.mcts_algorithm import MCTS

In [3]:
game = ConnectFourGame.as_new_game(verbose = True)
root_node = Node.as_root_node_from_game(
    game = game
)
print(len(root_node.untried_actions))  # should be 7
print(len(root_node.children)) 

7
0


In [4]:
game = ConnectFourGame.as_new_game(verbose = True)
seed = 0
rng = np.random.default_rng(seed=seed)

while game.result is None:
# for i in range(5):
    root_node = Node.as_root_node_from_game(game=game.copy())
    mcts = MCTS(
        root_node = root_node,
        rng = rng
    )
    optimal_action = mcts.run(n_iters = int(1e2))
    game.make_move(move = optimal_action.move)
    game.print_board()

. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . X . .
0 1 2 3 4 5 6
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . O . X . .
0 1 2 3 4 5 6
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . O . X X .
0 1 2 3 4 5 6
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . O . X X O
0 1 2 3 4 5 6
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. X O . X X O
0 1 2 3 4 5 6
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
O X O . X X O
0 1 2 3 4 5 6
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . X .
O X O . X X O
0 1 2 3 4 5 6
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . O .
. . . . . X .
O X O . X X O
0 1 2 3 4 5 6
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . O .
X . . . . X .
O X O . X X O
0 1 2 3 4 5 6
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . O .
X . . . O X .
O X O . X X O
0 1 2 3 4 5 6
. . . . . . .
. . . 

In [10]:
from collections import Counter
from joblib import Parallel, delayed
from tqdm import tqdm
import numpy as np

def play_game(n_iters: int, seed: int) -> int:
    from src.game.connect_four import ConnectFourGame
    from src.mcts.tree import Node
    from src.mcts.mcts_algorithm import MCTS
    import numpy as np

    rng = np.random.default_rng(seed)
    game = ConnectFourGame.as_new_game()
    
    while game.result is None:
        root = Node.as_root_node_from_game(game=game)
        mcts = MCTS(root_node=root, rng=rng)
        action = mcts.run(n_iters=n_iters)
        game.make_move(action.move)
    
    return game.result

if __name__ == '__main__':
    n_games = 1000
    n_iters = 1000
    n_workers = 16

    results_list = Parallel(n_jobs=n_workers)(
        delayed(play_game)(n_iters, seed) 
        for seed in tqdm(range(n_games), desc='Playing games')
    )

    results = Counter(results_list)
    n_played = sum(results.values())
    print(f"\nFinal results over {n_played} games:")
    print(f"X wins (1):  {results[1]}  ({results[1]/n_played:.1%})")
    print(f"O wins (-1): {results[-1]}  ({results[-1]/n_played:.1%})")
    print(f"Draws  (0):  {results[0]}  ({results[0]/n_played:.1%})")

Playing games: 100%|██████████| 1000/1000 [06:42<00:00,  2.48it/s]



Final results over 1000 games:
X wins (1):  456  (45.6%)
O wins (-1): 511  (51.1%)
Draws  (0):  33  (3.3%)


In [9]:
import time
from joblib import Parallel, delayed

for n_workers in [1, 2, 4, 8, 12, 16]:
    start = time.time()
    Parallel(n_jobs=n_workers)(
        delayed(play_game)(1000, seed) 
        for seed in range(16)
    )
    elapsed = time.time() - start
    print(f"Workers: {n_workers:2d} | Time: {elapsed:.1f}s | Games/sec: {32/elapsed:.1f}")

Workers:  1 | Time: 55.6s | Games/sec: 0.6
Workers:  2 | Time: 38.5s | Games/sec: 0.8
Workers:  4 | Time: 19.6s | Games/sec: 1.6
Workers:  8 | Time: 10.6s | Games/sec: 3.0
Workers: 12 | Time: 11.0s | Games/sec: 2.9
Workers: 16 | Time: 7.7s | Games/sec: 4.1
